# Husqvarna Marketplace — Analytical Layer

This notebook presents the analytical layer for the Cloud Data & AI Engineer take-home exercise. It answers all four required business questions and one additional question using the Gold star schema.

The analytical SQL is kept consistent with `analysis/analysis.sql`. The notebook is intended as the reviewer-facing analytical deliverable.

## Gold layer context

| Gold table | Rows |
|---|---:|
| `dim_customer` | 99,441 |
| `dim_seller` | 3,095 |
| `dim_product` | 32,951 |
| `dim_date` | 683 |
| `fact_order_item` | 112,650 |
| `fact_payment` | 103,886 |
| `fact_review` | 102,989 |

The analytical layer is built on these Gold Parquet datasets.

## Q1 — Seller × category combinations with the highest recent late-delivery rates

**Definition:** late delivery means `delivered_at > est_delivery_date`. “Recently” is defined as the last 90 days relative to the latest purchase date in the dataset. Combinations are limited to at least five delivered items to avoid ranking one-off combinations.

**Observed result:** the highest-ranked combinations in the current dataset reach a 100% late-delivery rate (6/6 delivered items for each of the top two combinations).

### Q1 SQL

```sql
-- Q1
WITH latest_date AS (
    SELECT MAX(CAST(purchased_at AS DATE)) AS max_purchase_date
    FROM fact_order_item
),
recent_delivery AS (
    SELECT foi.seller_key, foi.product_key, foi.delivered_at, foi.est_delivery_date
    FROM fact_order_item foi
    CROSS JOIN latest_date d
    WHERE CAST(foi.purchased_at AS DATE) >= DATE_SUB(d.max_purchase_date, 90)
      AND foi.delivered_at IS NOT NULL
      AND foi.est_delivery_date IS NOT NULL
)
SELECT ds.seller_id, dp.category_family, COUNT(*) AS delivered_items,
       SUM(CASE WHEN rd.delivered_at > rd.est_delivery_date THEN 1 ELSE 0 END) AS late_items,
       ROUND(100.0 * SUM(CASE WHEN rd.delivered_at > rd.est_delivery_date THEN 1 ELSE 0 END) / COUNT(*), 2) AS late_delivery_rate_pct
FROM recent_delivery rd
JOIN dim_seller ds ON rd.seller_key = ds.seller_key
JOIN dim_product dp ON rd.product_key = dp.product_key
WHERE dp.category_family IS NOT NULL
GROUP BY ds.seller_id, dp.category_family
HAVING COUNT(*) >= 5
ORDER BY late_delivery_rate_pct DESC, delivered_items DESC;
```

## Q2 — Average purchase-to-delivery lag by state

**Definition:** state is the customer/destination state. The metric is the average calendar-day difference between `purchased_at` and `delivered_at`, using delivered items with both timestamps present.

**Observed result:** AP and RR have the highest average lag in the current output at 28.22 and 28.17 days respectively; BA has 19.19 days across 3,683 delivered items.

### Q2 SQL

```sql
SELECT dc.customer_state,
       ROUND(AVG(DATEDIFF(CAST(foi.delivered_at AS DATE), CAST(foi.purchased_at AS DATE))), 2) AS average_purchase_to_delivery_days,
       COUNT(*) AS delivered_items
FROM fact_order_item foi
JOIN dim_customer dc ON foi.customer_key = dc.customer_key
WHERE foi.purchased_at IS NOT NULL AND foi.delivered_at IS NOT NULL
GROUP BY dc.customer_state
ORDER BY average_purchase_to_delivery_days DESC;
```

## Q3 — Sharpest month-over-month rise in negative reviews

**Definition:** negative reviews have a score ≤ 2. The comparison uses distinct `review_id` counts by month and category family; the ranking metric is percentage change from the previous month.

**Important interpretation:** very large percentages can occur when the previous month's count is very small. For example, Fashion rises from 1 negative review to 18 in August 2018, which produces a 1700% increase. The absolute change (17 reviews) should therefore be considered alongside the percentage.

### Q3 SQL

```sql
WITH monthly_negative_reviews AS (
    SELECT DATE_TRUNC('month', fr.review_creation_date) AS review_month,
           fr.category_family,
           COUNT(DISTINCT CASE WHEN fr.review_score <= 2 THEN fr.review_id END) AS negative_reviews
    FROM fact_review fr
    WHERE fr.review_score IS NOT NULL AND fr.category_family IS NOT NULL
    GROUP BY DATE_TRUNC('month', fr.review_creation_date), fr.category_family
),
with_previous_month AS (
    SELECT review_month, category_family, negative_reviews,
           LAG(negative_reviews) OVER (PARTITION BY category_family ORDER BY review_month)
               AS previous_month_negative_reviews
    FROM monthly_negative_reviews
)
SELECT review_month, category_family, negative_reviews, previous_month_negative_reviews,
       negative_reviews - previous_month_negative_reviews AS absolute_change,
       ROUND(100.0 * (negative_reviews - previous_month_negative_reviews)
             / previous_month_negative_reviews, 2) AS mom_change_pct
FROM with_previous_month
WHERE previous_month_negative_reviews > 0
ORDER BY mom_change_pct DESC;
```

## Q4 — Orders with anomalous value versus category p95

**Definition:** order-category value is `SUM(item_price + freight_cost)`. The p95 baseline is calculated separately for each category family. Orders above their category's p95 are reported.

**Modelling note:** multi-category orders are evaluated at the order × category level rather than forcing a single category onto the whole order.

### Q4 SQL

```sql
WITH order_category_value AS (
    SELECT foi.order_id, dp.category_family,
           SUM(foi.item_price + foi.freight_cost) AS order_category_value
    FROM fact_order_item foi
    JOIN dim_product dp ON foi.product_key = dp.product_key
    WHERE dp.category_family IS NOT NULL
    GROUP BY foi.order_id, dp.category_family
),
category_p95 AS (
    SELECT category_family,
           PERCENTILE_APPROX(order_category_value, 0.95) AS category_p95_value
    FROM order_category_value
    GROUP BY category_family
)
SELECT ocv.order_id, ocv.category_family,
       ROUND(ocv.order_category_value, 2) AS order_category_value,
       ROUND(cp.category_p95_value, 2) AS category_p95_value,
       ROUND(ocv.order_category_value / cp.category_p95_value, 2) AS multiple_of_category_p95
FROM order_category_value ocv
JOIN category_p95 cp ON ocv.category_family = cp.category_family
WHERE ocv.order_category_value > cp.category_p95_value
ORDER BY multiple_of_category_p95 DESC;
```

## Q5 — Additional question: which categories generate the most revenue and units?

This additional analysis ranks category families by total value (`item_price + freight_cost`) and also shows orders, units, item revenue and freight cost.

**Observed result:** Home & Living ranks first by total value at approximately 2.09M in the current output, followed by Beauty & Personal Care at approximately 1.89M and Furniture at approximately 1.42M.

### Q5 SQL

```sql
SELECT dp.category_family,
       COUNT(DISTINCT foi.order_id) AS orders,
       COUNT(*) AS units_sold,
       ROUND(SUM(foi.item_price), 2) AS item_revenue,
       ROUND(SUM(foi.freight_cost), 2) AS freight_cost,
       ROUND(SUM(foi.item_price + foi.freight_cost), 2) AS total_value
FROM fact_order_item foi
JOIN dim_product dp ON foi.product_key = dp.product_key
WHERE dp.category_family IS NOT NULL
GROUP BY dp.category_family
ORDER BY total_value DESC;
```

## Analytical takeaways

- Recent late-delivery performance is concentrated in specific seller × category combinations; the worst combinations in the current output have 100% late-delivery rates, although their sample sizes are only six delivered items.
- Delivery lag varies materially by destination state, with AP/RR at roughly 28 days in the current output.
- Negative-review MoM percentages need to be interpreted together with the underlying counts because small prior-month bases create very large percentages.
- The p95 analysis identifies extreme order-category values, including cases more than 40× the category p95 baseline.
- Home & Living and Beauty & Personal Care are the two highest-value category families in the current Q5 output.

## Reproducibility

The production analytical SQL remains in `analysis/analysis.sql`. `analysis/run_analysis.py` registers the Gold Parquet tables as Spark SQL views and executes the analytical statements. The notebook is the reviewer-facing presentation of that analytical layer.